# Credit Card Fraud Classfication using MLib for RDD

# Install Java and Spark on Hadoop

In [ ]:
import findspark
findspark.init()

In [ ]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.mllib.regression import LabeledPoint
from pyspark.mllib.linalg import Vectors
from pyspark.mllib.feature import StandardScaler
from pyspark.mllib.classification import LogisticRegressionWithSGD
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

In [ ]:
spark = SparkSession.builder \
    .appName("LowLevelDecisionTreeNYCTaxi") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

### Cào bộ dữ liệu từ trang của Kaggle sử dụng `cURL`

In [ ]:
!curl -L -o ./creditcardfraud.zip\
  https://www.kaggle.com/api/v1/datasets/download/mlg-ulb/creditcardfraud

!python -m zipfile -e 'creditcardfraud.zip' './'
!rm 'creditcardfraud.zip'

### Đọc và parse dữ liệu từ hdfs

In [ ]:
import os

lines = sc.textFile(f"file:///{os.getcwd()}/creditcard.csv")
cols = lines.first()
raw_data = lines.filter(lambda line: line != cols).cache()

### Lấy ra features và label dữ liệu

In [ ]:
cols = [col.replace('"','') for col in cols.split(",")]

### Lấy tập dữ liệu và chuẩn hóa kiểu dữ liệu

In [ ]:
data = raw_data.map(lambda line: [float(x.replace('"','').strip()) for x in line.split(",")]).cache()

In [ ]:
idx = [0,29]
cols = cols[1:29] + [cols[-1]]
pc_data = data.map(lambda row: tuple([row[i] for i in range(len(row)) if i not in idx])).cache()

### Kiểm tra trùng lặp dữ liệu

In [ ]:
# Cho biết các bản ghi có dữ liệu trùng
num_duplicate_data = pc_data.count()-pc_data.distinct().count()
duplicate_ratio = (num_duplicate_data)/pc_data.count() * 100
print(f"Dữ liệu trùng lặp: {num_duplicate_data}")
print(f"Tỉ lệ dữ liệu trùng lặp: {duplicate_ratio:.4f} %")
disc_pcdata = pc_data.distinct()

Dữ liệu trùng lặp: 9144
Tỉ lệ dữ liệu trùng lặp: 3.2106 %


### Chuyển đổi dữ liệu thành RDD của LabelPoint

In [ ]:
class_col_index = cols.index("Class")

def row_to_labeled_point(row):
    '''Chuyển đổi một hàng dữ liệu (từ RDD of tuples) thành đối tượng LabeledPoint cho MLlib.

    Args:
        row (tuple): Một hàng dữ liệu dưới dạng tuple.

    Returns:
        LabeledPoint: Đối tượng LabeledPoint với nhãn (label) và vector đặc trưng (dense vector).'''

    # Access the label using the determined index
    label = int(row[class_col_index])

    # Create the feature vector excluding the label and excluded columns
    feature_values = [float(row[i]) for i in range(len(row)) if i != class_col_index]

    return LabeledPoint(label, Vectors.dense(feature_values))


In [ ]:
rdd_lp = disc_pcdata.map(row_to_labeled_point).cache()

### Chuẩn hóa dữ liệu sang z-score

Các mô hình sử dụng gradient descent như hàm loss có xu hướng nhạy cảm nếu các feature có các khoảng phạm vi giá trị khác nha. Do đó bước chuẩn hóa dữ liệu là thiết yếu.

In [ ]:
train_rdd, test_rdd = rdd_lp.randomSplit([0.8, 0.2], seed=42)

In [ ]:
features_rdd = train_rdd.map(lambda lp: lp.features)

In [ ]:
scaler = StandardScaler(withMean=True, withStd=True)
scaler_model = scaler.fit(features_rdd)

In [ ]:
rdd_data = rdd_lp.map(lambda lp: lp.label).zip(scaler_model.transform(rdd_lp.map(lambda lp: lp.features)))
rdd_data = rdd_data.map(lambda x: LabeledPoint(x[0], x[1]))

### Chia tập train-test

In [ ]:
train_rdd, test_rdd = rdd_data.randomSplit([0.8, 0.2], seed=42)

## Huấn luyện mô hình

Ta sẽ sử dụng nhiều bộ tham số lr với iters để tìm bộ tham số tối ưu hóa hiệu suất mô hình

In [ ]:
learning_rates = [1, 0.01, 0.001]
listIters = [20,50,100]

In [ ]:
results = []
for lr, iters in zip(learning_rates,listIters):
    model = LogisticRegressionWithSGD.train(
            train_rdd,
            iterations=iters,
            step=lr,
            convergenceTol=1e-6
        )
    # Các thông số đánh giá
    prediction_and_labels = test_rdd.map(lambda lp: (float(model.predict(lp.features)), lp.label))

    # Tính AUC bằng BinaryClassificationMetrics
    metrics_binary = BinaryClassificationMetrics(prediction_and_labels)
    auc = metrics_binary.areaUnderROC

    # Tính Accuracy và Recall bằng MulticlassMetrics
    metrics_multi = MulticlassMetrics(prediction_and_labels)
    accuracy = metrics_multi.accuracy
    recall = metrics_multi.recall(label=1.0)  # Recall cho nhãn positive (1.0)

    TP = prediction_and_labels.filter(lambda pl: pl[0] == 1.0 and pl[1] == 1.0).count()
    TN = prediction_and_labels.filter(lambda pl: pl[0] == 0.0 and pl[1] == 0.0).count()
    FP = prediction_and_labels.filter(lambda pl: pl[0] == 1.0 and pl[1] == 0.0).count()
    FN = prediction_and_labels.filter(lambda pl: pl[0] == 0.0 and pl[1] == 1.0).count()

    results.append({
        "lr": lr,
        "iters": iters,
        "auc": auc,
        "accuracy": accuracy,
        "recall": recall,
        "prediction_result": {
            "TP": TP,
            "TN": TN,
            "FP": FP,
            "FN": FN
        }
    })

/home/nqthinh/hadoop/spark-3.5.5-bin-hadoop3/python/pyspark/mllib/classification.py:395: FutureWarning: Deprecated in 2.0.0. Use ml.classification.LogisticRegression or LogisticRegressionWithLBFGS.
  warnings.warn(
25/05/10 14:19:38 WARN BlockManager: Task 31 already completed, not releasing lock for rdd_17_0
25/05/10 14:19:38 WARN BlockManager: Task 31 already completed, not releasing lock for rdd_17_0
25/05/10 14:19:39 WARN BlockManager: Task 32 already completed, not releasing lock for rdd_17_0
25/05/10 14:19:39 WARN BlockManager: Task 32 already completed, not releasing lock for rdd_17_0
25/05/10 14:19:55 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
/home/nqthinh/hadoop/spark-3.5.5-bin-hadoop3/python/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(
25/05/10 14:20:00 WARN BlockManager: Task 149 already completed, not releasing lock for rdd_17_0
25/05/10 14:20:0

## Lưu kết quả

In [ ]:
def check_label_ratio(rdd):
    """
    Hàm kiểm tra tỉ lệ các nhãn trong RDD chứa LabeledPoint.
    Args:
        rdd: RDD chứa các LabeledPoint (có thuộc tính .label)
    Returns:
        Một danh sách các dictionary, mỗi dictionary chứa:
        - label: Nhãn
        - count: Số lượng mẫu của nhãn
        - ratio: Tỉ lệ của nhãn trong tổng số mẫu
    """
    # Đếm số lượng mẫu theo nhãn
    label_counts = rdd.map(lambda lp: (lp.label, 1)).reduceByKey(lambda a, b: a + b).collectAsMap()

    # Tính tổng số mẫu
    total = sum(label_counts.values())

    # Tính tỉ lệ và tạo danh sách kết quả
    result = [
        {"label": int(label), "count": count, "ratio": f"{(count / total * 100):.3f}%"}
        for label, count in label_counts.items()
    ]

    return result

In [ ]:
final_result = {
    "train": check_label_ratio(train_rdd),
    "test": check_label_ratio(test_rdd),
    "model_result": results
}

In [ ]:
import json
output_file = "./Mllib_RDD.json"
with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final_result, f, indent=4, ensure_ascii=False)

print(f"Kết quả đã được lưu vào file: {output_file}")

Kết quả đã được lưu vào file: ./Mllib_RDD.json
